# **Análise Exploratória de Dados da API GDC**
GDC: Genomic Data Commons

# Importação de Bibliotecas

In [1]:
import os
import re

import pandas as pd
import requests

# Constantes

In [2]:
# URL HTML base da API GDC
GDC_API_ENDPOINT = 'https://api.gdc.cancer.gov'

# Endpoint para download dos arquivos
DATA_ENDPOINT = f'{GDC_API_ENDPOINT}/data'

# Caminho da pasta de dados externos
EXTERNAL_DATA_PATH = '../../data/external/exploration'

# Caminho da pasta de dados intermediários
INTERIM_DATA_PATH = '../../data/interim/exploration'

# Carregamento de Dados

In [3]:
# DataFrame com os casos de interesse
df_cases = pd.read_csv(
    f'{INTERIM_DATA_PATH}/gdc-cases-of-interest.csv'
)

# DataFrame com os arquivos de interesse
df_files = pd.read_csv(
    f'{INTERIM_DATA_PATH}/gdc-files-of-interest.csv'
)

# Quantificação dos Arquivos

In [4]:
# Colunas usadas na agregação
columns = [
    'access',
    'experimental_strategy',
    'data_category',
    'data_type',
    'data_format'
]

# Agrega os dados sobre os arquivos e faz a contagem
df_files_agg = df_files \
    .groupby(columns) \
    .agg(count=pd.NamedAgg(column='file_id', aggfunc='count'))

# Faz a contagem normalizada dos dados dos arquivos
df_files_agg['%count'] = df_files_agg['count'] / df_files_agg['count'].sum()
df_files_agg['%count'] = (df_files_agg['%count'] * 100).round(2)

# Printa o resultado
df_files_agg

count  \
access     experimental_strategy data_category           data_type                         data_format          
controlled RNA-Seq               Sequencing Reads        Aligned Reads                     BAM          59324   
                                 Structural Variation    Transcript Fusion                 BEDPE        39118   
                                                                                           TSV          36983   
                                 Transcriptome Profiling Splice Junction Quantification    TSV          20202   
           miRNA-Seq             Sequencing Reads        Aligned Reads                     BAM          17989   
open       RNA-Seq               Transcriptome Profiling Gene Expression Quantification    TSV          20202   
           miRNA-Seq             Transcriptome Profiling Isoform Expression Quantification TSV           6328   
                                                                                           TXT          11661   
                                                         miRNA Expression Quantification   TSV           6328   
                                                                                           TXT          11661   

                                                                                                        %count  
access     experimental_strategy data_category           data_type                         data_format          
controlled RNA-Seq               Sequencing Reads        Aligned Reads                     BAM           25.82  
                                 Structural Variation    Transcript Fusion                 BEDPE         17.02  
                                                                                           TSV           16.09  
                                 Transcriptome Profiling Splice Junction Quantification    TSV            8.79  
           miRNA-Seq             Sequencing Reads        Aligned Reads                     BAM            7.83  
open       RNA-Seq               Transcriptome Profiling Gene Expression Quantification    TSV            8.79  
           miRNA-Seq             Transcriptome Profiling Isoform Expression Quantification TSV            2.75  
                                                                                           TXT            5.07  
                                                         miRNA Expression Quantification   TSV            2.75  
                                                                                           TXT            5.07

## Casos

In [5]:
# Define a condição de filtragem
condition = '(data_category == "Transcriptome Profiling") and (access == "open")'

# Filtra os arquivos e recupera as informações relativas aos casos
df_files_and_cases = df_files \
    .query(condition) \
    .reset_index(drop=True) \
    .merge(right=df_cases, on='case_id', how='inner')

### Origens Primárias por Caso

In [6]:
# Agrega os casos de acordo com suas origens primárias e faz a contagem
df_files_and_cases \
    .groupby(['case_id', 'primary_site']) \
    .agg(distinct_sites=pd.NamedAgg(column='primary_site', aggfunc='nunique')) \
    .sort_values(by='distinct_sites', ascending=False)

,,distinct_sites
case_id,primary_site,
0004d251-3f70-4395-b175-c94c2f5b1b81,Liver and intrahepatic bile ducts,1
ab45050e-2965-4d33-bff4-cd7a067ddd3d,Cervix uteri,1
aac5502e-147d-4ff9-9876-d9bffacf3dd0,Testis,1
aad167ab-4772-4616-afb3-265e00d4a3f0,Bladder,1
aad65d8f-47ba-5ad5-925e-5050b691f319,Hematopoietic and reticuloendothelial systems,1
...,...,...
54bacd1c-a6c0-5c28-9140-750e67395d87,Kidney,1
54c04bcc-ebd5-461e-bbd5-b4d791b00c95,Bladder,1
54c9850c-9b67-42ea-a04b-16a88f786dfa,Kidney,1


### Tipos de Doença por Caso

In [7]:
# Agrega os casos de acordo com seus tipos de doença e faz a contagem
df_files_and_cases \
    .groupby(['case_id', 'disease_type']) \
    .agg(distinct_diseases=pd.NamedAgg(column='disease_type', aggfunc='nunique')) \
    .sort_values(by='distinct_diseases', ascending=False)

,,distinct_diseases
case_id,disease_type,
0004d251-3f70-4395-b175-c94c2f5b1b81,Adenomas and Adenocarcinomas,1
ab45050e-2965-4d33-bff4-cd7a067ddd3d,Squamous Cell Neoplasms,1
aac5502e-147d-4ff9-9876-d9bffacf3dd0,Germ Cell Neoplasms,1
aad167ab-4772-4616-afb3-265e00d4a3f0,Transitional Cell Papillomas and Carcinomas,1
aad65d8f-47ba-5ad5-925e-5050b691f319,Myeloid Leukemias,1
...,...,...
54c04bcc-ebd5-461e-bbd5-b4d791b00c95,Transitional Cell Papillomas and Carcinomas,1
54c9850c-9b67-42ea-a04b-16a88f786dfa,Adenomas and Adenocarcinomas,1
54cf3bfe-51c0-4cca-9054-bf76b05a9371,"Cystic, Mucinous and Serous Neoplasms",1


### Arquivos por Caso

In [8]:
# Agrega os casos de acordo com os tipos de dados de seus arquivos e faz a contagem
df_files_and_cases \
    .groupby(['case_id', 'data_type']) \
    .agg(distinct_files=pd.NamedAgg(column='file_id', aggfunc='nunique')) \
    .sort_values(by='distinct_files', ascending=False)

distinct_files
case_id                              data_type                                        
842402de-519e-4588-ad49-19df18db899b Gene Expression Quantification                  8
3a4afbe7-60d6-4a0d-8166-56a04ff127b0 Gene Expression Quantification                  8
9ff6d022-6e23-4f44-a480-1b61929e6ee3 miRNA Expression Quantification                 6
                                     Isoform Expression Quantification               6
                                     Gene Expression Quantification                  6
...                                                                                ...
5c266025-e590-457c-af86-4c2dc9797267 Gene Expression Quantification                  1
                                     Isoform Expression Quantification               1
                                     miRNA Expression Quantification                 1
5c2b9799-4caa-407e-88fb-c198fec0bc83 Gene Expression Quantification                  1
fffdb1d9-58d1-425c-ac12-1e1e5f443bf7 miRNA Expression Quantification                 1

[47171 rows x 1 columns]

In [9]:
# Calcula a quantidade de casos com mais de um arquivo miRNA-Seq ou RNA-Seq associado
df_files_and_cases \
    .groupby(['case_id', 'data_type']) \
    .agg(distinct_files=pd.NamedAgg(column='file_id', aggfunc='nunique')) \
    .query('distinct_files > 1') \
    .sort_values(by='distinct_files', ascending=False) \
    .reset_index() \
    .nunique()

case_id           3322
data_type            3
distinct_files       6
dtype: int64

In [10]:
# Printa os registros de um caso específico
df_files_and_cases \
    .query('case_id == "9ff6d022-6e23-4f44-a480-1b61929e6ee3"') \
    .sort_values(by=['data_type']) \
    .reset_index(drop=True)

,case_id,data_format,access,file_id,data_type,data_category,experimental_strategy,primary_site,disease_type,project_id
0,9ff6d022-6e23-4f44-a480-1b61929e6ee3,TSV,open,ceb4dd9b-6f10-4358-8312-3b126952d3cc,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
1,9ff6d022-6e23-4f44-a480-1b61929e6ee3,TSV,open,c54a604e-9379-46f1-938c-e2d09f8538d8,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
2,9ff6d022-6e23-4f44-a480-1b61929e6ee3,TSV,open,d12e199e-8471-4f60-8da2-2b479db61ab4,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
3,9ff6d022-6e23-4f44-a480-1b61929e6ee3,TSV,open,167073fa-9e38-4f8d-af1f-301ed3a8b5f7,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
4,9ff6d022-6e23-4f44-a480-1b61929e6ee3,TSV,open,93c8678d-7afd-4be2-92cb-c39ad7701b43,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
5,9ff6d022-6e23-4f44-a480-1b61929e6ee3,TSV,open,b96247db-6f2a-4d26-9ac4-142e1c079e0e,Gene Expression Quantification,Transcriptome Profiling,RNA-Seq,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
6,9ff6d022-6e23-4f44-a480-1b61929e6ee3,TSV,open,c22a38cc-064b-46b8-a576-2fcbcfec7ceb,Isoform Expression Quantification,Transcriptome Profiling,miRNA-Seq,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
7,9ff6d022-6e23-4f44-a480-1b61929e6ee3,TSV,open,229a4e0b-1bf1-4432-9cda-864dcbfaea55,Isoform Expression Quantification,Transcriptome Profiling,miRNA-Seq,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
8,9ff6d022-6e23-4f44-a480-1b61929e6ee3,TSV,open,2d33a6eb-3db9-49e4-85f2-5a3387c21cf5,Isoform Expression Quantification,Transcriptome Profiling,miRNA-Seq,Kidney,Adenomas and Adenocarcinomas,CPTAC-3
9,9ff6d022-6e23-4f44-a480-1b61929e6ee3,TSV,open,0afd73ac-8fbc-418f-9d58-b1da06da4c98,Isoform Expression Quantification,Transcriptome Profiling,miRNA-Seq,Kidney,Adenomas and Adenocarcinomas,CPTAC-3


### Arquivos por Origem Primária

In [11]:
# Agrega os arquivos de acordo com a origem primária e faz a contagem
df_site_agg = df_files_and_cases \
    .groupby('primary_site') \
    .agg(count=pd.NamedAgg(column='file_id', aggfunc='count')) \
    .sort_values(by='count', ascending=False) \
    .reset_index()

# Printa o resultado
df_site_agg

,primary_site,count
0,Hematopoietic and reticuloendothelial systems,9707
1,Kidney,5468
2,Bronchus and lung,5433
3,Thyroid gland,4630
4,Breast,4022
5,Brain,2767
6,Colon,2033
7,Corpus uteri,1734
8,Prostate gland,1656
9,Ovary,1653


# Download de Arquivos

In [12]:
# UUIDs dos arquivos de interesse
file_ids = [
    'b96247db-6f2a-4d26-9ac4-142e1c079e0e', # Gene Expression Quantification
    '0cf6cded-942f-4141-a4a5-35afb7082f37', # Isoform Expression Quantification
    '01e3d493-7e2a-4a50-b8cb-2597143a8e1a'  # miRNA Expression Quantification
]


for file_id in file_ids:
    # Requisita o download do arquivo ao endpoint 
    response = requests.get(
        url=f'{DATA_ENDPOINT}/{file_id}', 
        headers={'Content-Type': 'application/json'}
    )

    # Obtém o nome do arquivo por meio da resposta do endpoint
    response_head_cd = response.headers['Content-Disposition']
    file_name = re.findall('filename=(.+)', response_head_cd)[0]

    # Armazena o arquivo na pasta de dados externos
    file_path = os.path.join(EXTERNAL_DATA_PATH, file_name)
    with open(file_path, 'wb') as output_file:
        output_file.write(response.content)